# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nando135/flyrank_ml_1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



Finding 1 — PCA Cluster Projection
The paper uses PCA to visualize the K-Means clusters and notes that overlap means the cluster boundaries are not perfectly clear.

Methodology question:
Were the clusters also evaluated in the full feature space? Since PCA keeps only part of the variance, a metric such as silhouette score would give stronger evidence of cluster separation.

Finding 2 — AI Model Performance
The paper compares OpenAI and Gemini within the same age groups and finds that neither consistently performs best.

Methodology question:
Besides age, were factors such as client, topic, intent, or editing process also controlled? If not, some performance differences may come from those factors rather than the model family itself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

baseline_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""

baseline_df = con.sql(baseline_query).df()

page_df = (
    baseline_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        sessions=("ga4_sessions", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        total_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
)

page_df["ctr"] = (
    page_df["clicks"] / page_df["impressions"]
).fillna(0)

page_df["engagement_rate"] = (
    page_df["engaged_sessions"] / page_df["sessions"]
).fillna(0)

print("Pages:", len(page_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 63856


In [5]:
feature_cols = [
    "impressions",
    "ctr",
    "sessions",
    "engagement_rate"
]

random_train, random_val = train_test_split(
    page_df,
    test_size=0.20,
    random_state=42
)

random_train_features = random_train[feature_cols].copy()
random_val_features = random_val[feature_cols].copy()

random_train_features["impressions"] = np.log1p(
    random_train_features["impressions"]
)
random_train_features["sessions"] = np.log1p(
    random_train_features["sessions"]
)

random_val_features["impressions"] = np.log1p(
    random_val_features["impressions"]
)
random_val_features["sessions"] = np.log1p(
    random_val_features["sessions"]
)

random_scaler = StandardScaler()

X_random_train = random_scaler.fit_transform(
    random_train_features
)

X_random_val = random_scaler.transform(
    random_val_features
)

random_model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

random_model.fit(X_random_train)

random_val_clusters = random_model.predict(
    X_random_val
)

random_silhouette = silhouette_score(
    X_random_val,
    random_val_clusters
)

random_overlap = (
    set(random_train["client_hash_id"])
    & set(random_val["client_hash_id"])
)

print("Random split validation silhouette:", random_silhouette)
print("Clients appearing in both random splits:", len(random_overlap))

Random split validation silhouette: 0.4830005654597387
Clients appearing in both random splits: 34


In [6]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        page_df,
        groups=page_df["client_hash_id"]
    )
)

group_train = page_df.iloc[train_idx].copy()
group_val = page_df.iloc[val_idx].copy()

group_train_features = group_train[feature_cols].copy()
group_val_features = group_val[feature_cols].copy()

group_train_features["impressions"] = np.log1p(
    group_train_features["impressions"]
)
group_train_features["sessions"] = np.log1p(
    group_train_features["sessions"]
)

group_val_features["impressions"] = np.log1p(
    group_val_features["impressions"]
)
group_val_features["sessions"] = np.log1p(
    group_val_features["sessions"]
)

group_scaler = StandardScaler()

X_group_train = group_scaler.fit_transform(
    group_train_features
)

X_group_val = group_scaler.transform(
    group_val_features
)

group_model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

group_model.fit(X_group_train)

group_val_clusters = group_model.predict(
    X_group_val
)

group_silhouette = silhouette_score(
    X_group_val,
    group_val_clusters
)

group_overlap = (
    set(group_train["client_hash_id"])
    & set(group_val["client_hash_id"])
)

print("Grouped split validation silhouette:", group_silhouette)
print("Clients appearing in both grouped splits:", len(group_overlap))

Grouped split validation silhouette: 0.4659947653606159
Clients appearing in both grouped splits: 0


In [7]:
comparison_df = pd.DataFrame({
    "split": [
        "Random split",
        "Client-grouped split"
    ],
    "validation_silhouette": [
        random_silhouette,
        group_silhouette
    ],
    "client_overlap": [
        len(random_overlap),
        len(group_overlap)
    ]
})

display(comparison_df)

,split,validation_silhouette,client_overlap
0,Random split,0.483001,34
1,Client-grouped split,0.465995,0


The random split achieved a validation silhouette score of 0.483, while the client-grouped split achieved 0.466. The random split had all 34 clients appearing across both training and validation, whereas the grouped split had zero client overlap. The small drop in silhouette suggests that some client-specific similarity was present in the random split, but the cluster structure remained reasonably stable when tested on unseen clients. Therefore, the grouped split provides a more honest validation result.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The final clustering features were impressions, CTR, sessions, and engagement rate. No product flags, baseline scores, cluster labels, or other model outputs were used as input features. All features were calculated from the same March 2026 observation window, so there is no future-window leakage within this analysis. Because K-Means is unsupervised, there is also no target label that could leak directly into the features. The main limitation is that CTR and engagement rate are derived from impressions/clicks and sessions/engaged sessions, so these features are related rather than fully independent.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In the March 2026 data, we observed that pages with higher visibility did not always show proportionally stronger CTR or engagement, which motivated the Week 4 review-priority baseline. In Week 5, K-Means identified four content groups with different visibility, CTR, session, and engagement patterns. The model achieved a validation silhouette score of 0.466 on unseen clients, compared with 0.483 under a random split. Because the grouped split had zero client overlap while the random split included all 34 clients in both sets, the grouped result provides the more conservative validation estimate. The clusters therefore appear reasonably stable across unseen clients, but they should be treated as descriptive decision-support patterns rather than fixed content categories or causal relationships.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.